# Nutrition5k: experimentos con transformers

Igual que `notebooks/mobilenet/02_experiments.ipynb` (mismas particiones, métricas, pérdida, augmentación y receta por
defecto), con los transformers de `src/nutrition5k/models/transformer.py`:

| Clave | Modelo | Por qué |
|---|---|---|
| `deit_t` | DeiT-Tiny (ViT puro, 5.5M, ImageNet-1k) | Atención frente a convolución con el tamaño y preentreno de MobileNetV3 |
| `fastvit_sa12` | FastViT-SA12 (10.6M, ImageNet-1k): RepMixer + autoatención en la última etapa | Candidato edge, híbrido real |
| `dinov2_s` | DINOv2 ViT-S/14 (21.6M, autosupervisado en LVD-142M) | Efecto del preentreno |
| `swin_t` | Swin-T (jerárquico, 27.5M, ImageNet-1k) | Familia de Swin-Nutrition; referencia de precisión |

- Notebook fino: solo llama a `python -m nutrition5k ...`. Los runs se guardan en `outputs/transformer/`.
- Todos los transformers se entrenan a 224 px (`train` rechaza otra resolución).
- `--unfreeze N` entrena los últimos N bloques en DeiT y DINOv2 (12 bloques) y las últimas N etapas en FastViT y Swin (4).
- Las decisiones se toman con **validación** en `03_results.ipynb` de esta carpeta, nunca con test.


## Parámetros

In [ ]:
import os

FULL = True          # False: piloto 32/8 para pruebas rápidas
MODE = "complete"    # "complete" (cada 5º frame, todas las cámaras) o "minimal" (1 frame por plato)
EPOCHS = 80
WORKERS = min(15, os.cpu_count()-2)  # procesos que decodifican JPEG

FULL_FLAG = "--full" if FULL else ""
SPLIT = "full" if FULL else "pilot"

## Preparación del entorno

In [ ]:
import os
import sys
from pathlib import Path

# The volume is mounted at /workspace only on RunPod.
RUNPOD = Path("/workspace").is_dir()
if RUNPOD:
    # SSH sessions (VS Code Remote) do not inherit the pod's variables, such as the W&B secret.
    for variable in Path("/proc/1/environ").read_text().split("\0"):
        name, _, value = variable.partition("=")
        if name == "WANDB_API_KEY":
            os.environ.setdefault(name, value)
    # The template points the Hugging Face cache at the volume, where its symlinks break
    # (object storage); keep the pretrained weights on the pod's local disk instead.
    os.environ["HF_HOME"] = "/root/.cache/huggingface"

REPOSITORY_URL = "https://github.com/victorabellotorres/thesis-multimodal-health-ai.git"
# Local disk of the pod: fast, but lost when the pod ends.
REPOSITORY_DIR = Path("/root/thesis-multimodal-health-ai")
BRANCH = "main"
# Persistent storage: the volume mounted at /workspace.
PERSIST_ROOT = Path("/workspace/nutrition5k")

if RUNPOD:
    if (REPOSITORY_DIR / ".git").is_dir():
        !git -C {REPOSITORY_DIR} fetch
        !git -C {REPOSITORY_DIR} checkout {BRANCH}
        !git -C {REPOSITORY_DIR} pull --ff-only
    else:
        !git clone -b {BRANCH} {REPOSITORY_URL} {REPOSITORY_DIR}
    %cd {REPOSITORY_DIR}
    %pip install -q -e .[train]
elif Path.cwd().parent.name == "notebooks":
    # Local: the package is already installed in code/.venv; run from the repository root.
    os.chdir("../..")

# The kernel's interpreter, so `!` commands use the same environment locally (VS Code may not activate .venv).
PYTHON = sys.executable

if RUNPOD:
    # Results live on the volume so nothing is lost when the pod ends.
    (PERSIST_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
    if not Path("outputs").exists():
        Path("outputs").symlink_to(PERSIST_ROOT / "outputs")
print("W&B:", "on" if os.environ.get("WANDB_API_KEY") else "off (only local files)")

En local, `fetch` y `prepare` trabajan directamente sobre `data/` y se saltan lo que ya está descargado.

En RunPod, si el `.tar` ya existe en el volumen se restaura; si no, `prepare` descarga los vídeos por lotes de 100 platos, extrae los frames reducidos a 320 px y borra los vídeos de cada lote. Después se guarda el `.tar` (solo frames) en el volumen para el próximo pod. En RunPod lo normal es subir el `.tar` preparado en el portátil (ver el README), así la GPU no espera a las descargas.

Si `prepare` falla (por ejemplo, un timeout que persiste tras los reintentos), no se crea el `.tar`. Si `prepare` falla o se interrumpe pero el pod sigue encendido, volver a ejecutar la celda continúa donde se quedó. Si el pod se para antes de crear el `.tar`, el disco local se pierde y la preparación empieza de nuevo.

In [ ]:
DATA_TAR = PERSIST_ROOT / f"data-{SPLIT}-{MODE}.tar"

if RUNPOD and DATA_TAR.is_file():
    !tar -xf {DATA_TAR}
else:
    !{PYTHON} -m nutrition5k fetch
    !{PYTHON} -m nutrition5k prepare --mode {MODE} {FULL_FLAG}
    if RUNPOD and _exit_code == 0:  # a failed prepare must not be cached as a partial .tar
        !tar -cf {DATA_TAR} --exclude='*.h264' data  # frames only, not the videos
!{PYTHON} -m nutrition5k preprocess  # excluded_dishes.csv: label errors left out of train and validation
!{PYTHON} -m nutrition5k check --mode {MODE} {FULL_FLAG}

## Baseline (media del conjunto de entrenamiento)

In [ ]:
!{PYTHON} -m nutrition5k baseline {FULL_FLAG}

In [ ]:
def train(backbone, unfreeze="all", heads="single", hidden=(512,), backbone_lr=1e-4, seed=0, all_seeds=False):
    """Run one configuration at 224 px; finished runs are skipped, interrupted ones resume."""
    hidden_args = " ".join(f"--hidden {size}" for size in (hidden or (0,)))
    command = (
        f"{PYTHON} -m nutrition5k train --backbone {backbone} --unfreeze {unfreeze} "
        f"--heads {heads} {hidden_args} --backbone-lr {backbone_lr} --epochs {EPOCHS} "
        f"--num-workers {WORKERS} --seed {seed} --mode {MODE} {FULL_FLAG}"
    )
    if all_seeds:
        command += " --all-seeds"
    print(command)
    get_ipython().system(command)

## Etapa 1: comparación de backbones

Configuración por defecto, la misma que en la etapa 1 de MobileNet: fine-tuning completo, cabeza `single`, capa oculta
compartida de 512, 224 px, lr del backbone 1e-4. Además, `dinov2_s` con el backbone congelado (`unfreeze 0`), porque
sus features están pensadas para usarse congeladas.

In [ ]:
for backbone in ("deit_t", "fastvit_sa12", "dinov2_s", "swin_t"):
    train(backbone)
# DINOv2 is meant to work frozen; without this run, stage 1 could discard it for the full fine-tuning alone.
train("dinov2_s", unfreeze=0)

## Etapa 2: ablaciones secuenciales sobre cada backbone elegido

Igual que en MobileNet: cada backbone de `BACKBONES` sigue su propio camino greedy, se elige por validación en
`03_results.ipynb` y el ganador se escribe en el `BEST_*` del paso siguiente. Empates (< ~1 punto): semillas 1 y 2 o la
opción más simple. `BACKBONES` está vacío hasta decidir la etapa 1, así que estas celdas no entrenan nada por accidente.

Solo dos pasos, por el presupuesto (GEP: ~6 configuraciones): profundidad de fine-tuning y lr del backbone. Cabezas,
tronco y resolución se quedan en el valor por defecto.

### Paso 1: profundidad del fine-tuning (0 = backbone congelado, N = últimos N bloques o etapas, all = completo)

In [ ]:
BACKBONES = ()  # elegir tras la etapa 1, p. ej. ("fastvit_sa12", "dinov2_s")
# DeiT y DINOv2 tienen 12 bloques; FastViT y Swin, 4 etapas.
UNFREEZE_OPTIONS = {"deit_t": (0, 3, 6), "dinov2_s": (0, 3, 6), "fastvit_sa12": (0, 1, 2), "swin_t": (0, 1, 2)}

for backbone in BACKBONES:
    for unfreeze in UNFREEZE_OPTIONS[backbone]:
        train(backbone, unfreeze=unfreeze)

### Paso 2: lr del backbone

Hipótesis (no verificada en este dataset): un ViT preentrenado pierde menos de lo aprendido con un lr más bajo al
hacer fine-tuning. Con el backbone congelado (`unfreeze = 0`) este paso no tiene sentido y se salta.

In [ ]:
BEST_UNFREEZE = {}  # ganador del paso 1, p. ej. {"fastvit_sa12": "all", "dinov2_s": 3}

for backbone in BACKBONES:
    if BEST_UNFREEZE[backbone] != 0:
        train(backbone, BEST_UNFREEZE[backbone], backbone_lr=3e-5)

## Etapa 3: semillas

Repetir la configuración final de cada backbone con las semillas fijas 0, 1 y 2 (la 0 ya está hecha y se reutiliza).
`--all-seeds` escribe la media ± SD en `outputs/transformer/<full|pilot>/seeds/<configuración>.json`.

In [ ]:
BEST_BACKBONE_LR = {}  # ganador del paso 2, p. ej. {"fastvit_sa12": 1e-4, "dinov2_s": 3e-5}

for backbone in BACKBONES:
    train(backbone, BEST_UNFREEZE[backbone], backbone_lr=BEST_BACKBONE_LR.get(backbone, 1e-4), all_seeds=True)